# 03 — Classical ML Models (v2) - 8-Model Optimized Training
## SOH Regression: Intra-Battery Chronological Split

**v2.0 Pipeline:**
- Load preprocessed `battery_features.csv` from NB02
- Intra-battery 80/20 chronological split per battery
- Train 8 core models with optimized hyperparameters
- Target: ≥95% within-±5% SOH accuracy on all models
- Save artifacts to `artifacts/v2/`

**Models (8 total):**
1. ExtraTrees (tree-based, unstained)
2. GradientBoosting (sequential ensemble)
3. RandomForest (bagging ensemble)
4. XGBoost (boosted trees with tuning)
5. LightGBM (fast gradient boosting)
6. SVR (support vector regression)
7. Ridge (linear with L2)
8. KNN-5 (instance-based with distance weighting)

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import joblib
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.svm import SVR
from sklearn.ensemble import (
    RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor
)
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import r2_score, mean_absolute_error
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

print('Setup complete.')

In [ ]:
# Setup v2 paths
WORKSPACE = Path('..').resolve()
V2_RESULTS = WORKSPACE / 'artifacts/v2/results'
V2_MODELS = WORKSPACE / 'artifacts/v2/models/classical'
V2_SCALERS = WORKSPACE / 'artifacts/v2/scalers'

print(f'V2 Results: {V2_RESULTS}')
print(f'V2 Models: {V2_MODELS}')
print(f'V2 Scalers: {V2_SCALERS}')

In [ ]:
# Load preprocessed features from NB02
features_df = pd.read_csv(V2_RESULTS / 'battery_features.csv')
print(f'Dataset shape: {features_df.shape}')
print(f'Batteries: {sorted(features_df["battery_id"].unique())}')
print(f'SOH range: {features_df["SoH"].min():.1f}% — {features_df["SoH"].max():.1f}%')

In [ ]:
# Intra-battery 80/20 split (by cycle number chronologically per battery)
train_parts = []
test_parts = []

for bid in sorted(features_df['battery_id'].unique()):
    bat_df = features_df[features_df['battery_id'] == bid].sort_values('cycle_number')
    cut_idx = int(len(bat_df) * 0.8)
    train_parts.append(bat_df.iloc[:cut_idx])
    test_parts.append(bat_df.iloc[cut_idx:])

train_df = pd.concat(train_parts, ignore_index=True)
test_df = pd.concat(test_parts, ignore_index=True)

print(f'Train: {len(train_df)} samples')
print(f'Test: {len(test_df)} samples')
print(f'Train SOH: {train_df["SoH"].min():.1f}% — {train_df["SoH"].max():.1f}%')
print(f'Test SOH: {test_df["SoH"].min():.1f}% — {test_df["SoH"].max():.1f}%')

In [ ]:
# Define feature columns (engineered from NB02)
feature_cols = [
    'cycle_number', 'ambient_temperature', 'peak_voltage', 'min_voltage',
    'voltage_range', 'avg_current', 'avg_temp', 'temp_rise', 'cycle_duration',
    'Re', 'Rct', 'delta_capacity'
]

X_train = train_df[feature_cols].values
y_train = train_df['SoH'].values
X_test = test_df[feature_cols].values
y_test = test_df['SoH'].values

print(f'X_train: {X_train.shape}')
print(f'y_train: {y_train.shape}')

In [ ]:
# Fit StandardScaler on training data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Save scaler
joblib.dump(scaler, V2_SCALERS / 'standard_scaler.joblib')
print('Scaler saved.')

In [ ]:
def evaluate_model(model_name, model, X_eval, y_eval, save_path=None):
    """Evaluate model and optionally save it."""
    y_pred = model.predict(X_eval)
    
    r2 = r2_score(y_eval, y_pred)
    mae = mean_absolute_error(y_eval, y_pred)
    within_5pct = float((np.abs(y_pred - y_eval) <= 5).mean() * 100)
    
    status = '✓ PASS' if within_5pct >= 95.0 else '✗ FAIL'
    print(f'{model_name:20s} | R²={r2:.4f} | MAE={mae:.2f} | Within-5%={within_5pct:.1f}% | {status}')
    
    if save_path:
        joblib.dump(model, save_path)
    
    return y_pred, {'model': model_name, 'r2': r2, 'mae': mae, 'within_5pct': within_5pct}

In [ ]:
# ExtraTrees (unscaled)
model_et = ExtraTreesRegressor(
    n_estimators=1000,
    min_samples_leaf=2,
    max_features=0.7,
    random_state=42,
    n_jobs=-1
)
model_et.fit(X_train, y_train)
_, metrics_et = evaluate_model('ExtraTrees', model_et, X_test, y_test,
                               V2_MODELS / 'extra_trees.joblib')

In [ ]:
# GradientBoosting (unscaled)
model_gb = GradientBoostingRegressor(
    n_estimators=800,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    random_state=42
)
model_gb.fit(X_train, y_train)
_, metrics_gb = evaluate_model('GradientBoosting', model_gb, X_test, y_test,
                               V2_MODELS / 'gradient_boosting.joblib')

In [ ]:
# RandomForest (unscaled)
model_rf = RandomForestRegressor(
    n_estimators=1000,
    min_samples_leaf=2,
    max_features=0.7,
    random_state=42,
    n_jobs=-1
)
model_rf.fit(X_train, y_train)
_, metrics_rf = evaluate_model('RandomForest', model_rf, X_test, y_test,
                               V2_MODELS / 'random_forest.joblib')

In [ ]:
# XGBoost (unscaled, tuned hyperparameters)
model_xgb = XGBRegressor(
    n_estimators=1200,
    max_depth=9,
    learning_rate=0.02,
    subsample=0.85,
    colsample_bytree=0.85,
    random_state=42,
    n_jobs=-1,
    verbosity=0
)
model_xgb.fit(X_train, y_train)
_, metrics_xgb = evaluate_model('XGBoost', model_xgb, X_test, y_test,
                                V2_MODELS / 'xgboost.joblib')

In [ ]:
# LightGBM (unscaled, tuned hyperparameters)
model_lgbm = LGBMRegressor(
    n_estimators=1200,
    num_leaves=127,
    learning_rate=0.02,
    subsample=0.85,
    colsample_bytree=0.85,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)
model_lgbm.fit(X_train, y_train)
_, metrics_lgbm = evaluate_model('LightGBM', model_lgbm, X_test, y_test,
                                 V2_MODELS / 'lightgbm.joblib')

In [ ]:
# SVR (scaled)
model_svr = SVR(
    C=1000.0,
    epsilon=0.1,
    kernel='rbf'
)
model_svr.fit(X_train_scaled, y_train)
_, metrics_svr = evaluate_model('SVR', model_svr, X_test_scaled, y_test,
                               V2_MODELS / 'svr.joblib')

In [ ]:
# Ridge (scaled)
model_ridge = Ridge(
    alpha=0.1
)
model_ridge.fit(X_train_scaled, y_train)
_, metrics_ridge = evaluate_model('Ridge', model_ridge, X_test_scaled, y_test,
                                  V2_MODELS / 'ridge.joblib')

In [ ]:
# KNN-5 (scaled, with distance weighting)
model_knn5 = KNeighborsRegressor(
    n_neighbors=5,
    weights='distance',
    n_jobs=-1
)
model_knn5.fit(X_train_scaled, y_train)
_, metrics_knn5 = evaluate_model('KNN-5', model_knn5, X_test_scaled, y_test,
                                V2_MODELS / 'knn_k5.joblib')

In [ ]:
# Collect results
all_metrics = [
    metrics_et, metrics_gb, metrics_rf, metrics_xgb, metrics_lgbm,
    metrics_svr, metrics_ridge, metrics_knn5
]

results_df = pd.DataFrame(all_metrics)
results_df = results_df.sort_values('within_5pct', ascending=False)

print('\n' + '='*70)
print('FINAL RESULTS - 8 MODEL COMPARISON')
print('='*70)
print(results_df.to_string(index=False))

# Count passes
n_passed = (results_df['within_5pct'] >= 95.0).sum()
print(f'\nPassed (≥95%): {n_passed}/8')

# Save results
results_df.to_csv(V2_RESULTS / 'v2_classical_results.csv', index=False)
print(f'\nResults saved to {V2_RESULTS / "v2_classical_results.csv"}')